
# Spam Email Detection — End‑to‑End Notebook
This notebook walks you through the complete workflow to build a spam classifier using the **UCI Spambase** dataset:
1) Setup & imports  
2) Load data (from web or local)  
3) Exploratory checks & preprocessing  
4) Train/test split  
5) Train four models (Naive Bayes, Logistic Regression, Decision Tree, Random Forest)  
6) Evaluate (accuracy, confusion matrix, classification report)  
7) Cross‑validation & basic tuning (optional)  
8) Feature importances & plots  
9) Save key results for your report


In [ ]:

# ==== 1) Setup & Imports ====
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def hr(title):
    print("\n" + "="*len(title))
    print(title)
    print("="*len(title))



## 2) Load the Spambase dataset
**Option A (Recommended):** Download directly from UCI.  
**Option B:** Load a local `spambase.csv` placed in the same folder.


In [ ]:

# ==== 2) Load Data (Web or Local) ====
USE_WEB = True  # set to False if you want to load from a local CSV

if USE_WEB:
    import urllib.request
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data"
    local_path = "spambase.csv"
    urllib.request.urlretrieve(url, local_path)
    print(f"Downloaded to: {local_path}")
else:
    local_path = "spambase.csv"
    assert os.path.exists(local_path), "Place spambase.csv in this directory or set USE_WEB=True."

import pandas as pd
df = pd.read_csv(local_path, header=None)
df.head()


In [ ]:

# Dimensions and quick checks
hr("Dataset Shape & NaNs")
print("Shape (rows, cols):", df.shape)
print("Total missing values:", df.isnull().sum().sum())

hr("Label Balance")
print(df.iloc[:, -1].value_counts())



## 3) Preprocess
- Separate features/label  
- Min‑Max scale features


In [ ]:

X = df.iloc[:, :-1].values   # features
y = df.iloc[:, -1].values    # label

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
print("Features scaled to [0,1].")



## 4) Train/Test Split


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)



## 5) Train Four Models


In [ ]:

models = {
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=2000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append((name, acc, model))
    print(f"{name:>18}: accuracy = {acc:.4f}")

results = sorted(results, key=lambda t: t[1], reverse=True)
best_name, best_acc, best_model = results[0]
hr(f"Best so far: {best_name} (accuracy {best_acc:.4f})")



## 6) Evaluate Best Model


In [ ]:

y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

fig = plt.figure(figsize=(5,4))
plt.imshow(cm, cmap='Blues')
plt.title(f"{best_name} — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
for (i, j) in [(i, j) for i in range(cm.shape[0]) for j in range(cm.shape[1])]:
    plt.text(j, i, int(cm[i, j]), ha='center', va='center')
plt.colorbar()
plt.show()

print(classification_report(y_test, y_pred))

try:
    if hasattr(best_model, "predict_proba"):
        y_proba = best_model.predict_proba(X_test)[:, 1]
        RocCurveDisplay.from_predictions(y_test, y_proba)
        plt.title(f"{best_name} — ROC Curve")
        plt.show()
        print("ROC AUC:", roc_auc_score(y_test, y_proba))
except Exception as e:
    print("ROC not available:", e)


 
## 7) Cross‑Validation (Optional ) I was trying out different things with this one



In [ ]:

from sklearn.base import clone
cv_model = clone(best_model)
scores = cross_val_score(cv_model, X_scaled, y, cv=5)
hr("5-Fold Cross-Validation")
print("Scores:", np.round(scores, 4))
print("Mean accuracy:", scores.mean())



## 8) Basic Hyperparameter Tuning (Optional)


In [ ]:

if isinstance(best_model, RandomForestClassifier):
    from sklearn.model_selection import GridSearchCV
    param_grid = {
        "n_estimators": [100, 200, 400],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5, 10]
    }
    grid = GridSearchCV(RandomForestClassifier(random_state=42),
                        param_grid, cv=3, n_jobs=-1)
    grid.fit(X_train, y_train)
    hr("Grid Search Best Params")
    print(grid.best_params_)
    tuned = grid.best_estimator_
    y_pred_tuned = tuned.predict(X_test)
    print("Tuned accuracy:", accuracy_score(y_test, y_pred_tuned))
else:
    print("Skip tuning: best model is not RandomForest.")



## 9) Feature Importances (Tree Models)


In [ ]:

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    idx_sorted = np.argsort(importances)[::-1][:15]
    plt.figure(figsize=(8,4))
    plt.bar(range(len(idx_sorted)), importances[idx_sorted])
    plt.xticks(range(len(idx_sorted)), idx_sorted, rotation=45)
    plt.title(f"{best_name} — Top 15 Feature Importances (by index)")
    plt.xlabel("Feature Index")
    plt.ylabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    print("Feature importances not available for this model.")



## 10) Save Artifacts for the Report



In [ ]:

acc_df = pd.DataFrame({
    "Model": [name for name, _, _ in results],
    "Accuracy": [float(f"{acc:.4f}") for _, acc, _ in results]
})
acc_df.to_csv("accuracy_summary.csv", index=False)

cm_df = pd.DataFrame(cm, columns=["Pred 0","Pred 1"], index=["Actual 0","Actual 1"])
cm_df.to_csv("confusion_matrix_best_model.csv")

report_text = classification_report(y_test, y_pred)
with open("classification_report_best_model.txt", "w") as f:
    f.write(report_text)

hr("Saved Artifacts")
print("accuracy_summary.csv")
print("confusion_matrix_best_model.csv")
print("classification_report_best_model.txt")



---
### What to write in your report
- **Task:** Binary classification (spam vs ham).  
- **Data:** UCI Spambase (4601 rows, 57 features + 1 label).  
- **Best model (example):** Random Forest with ~0.94 accuracy.  
- **Validation:** 5-fold CV average accuracy.  
- **Future work:** Try SVM; tune thresholds to trade precision vs recall; experiment with TF‑IDF from raw text.
